# Unified preprocessing: hex metadata + connectivity + pruning

Replaces notebooks 01–04. Three phases, all from local compressed NC files:

- **Phase 1**: Scan all 9 NC files (metadata arrays only) → canonical hex mapping
- **Phase 2**: Scan all 9 NC files (obs arrays) → connectivity weights
- **Phase 3**: Prune NaN-corner hexes + spatially isolated clusters

**Memory constraint**: only one NC file open at a time.

In [1]:
from pathlib import Path
import numpy as np

# Resolve input_comp/ relative to wherever the kernel runs from
_cwd = Path.cwd()
if (_cwd / "input_comp").exists():
    LOCAL_INPUT_DIR = (_cwd / "input_comp").resolve()
elif (_cwd / "../input_comp").exists():
    LOCAL_INPUT_DIR = (_cwd / "../input_comp").resolve()
else:
    raise FileNotFoundError(f"Cannot find input_comp/ from {_cwd}")

# Resolve database/data/ output directory
if (_cwd / "../database/data").exists():
    OUT_DIR = (_cwd / "../database/data").resolve()
elif (_cwd / "../../database/data").exists():
    OUT_DIR = (_cwd / "../../database/data").resolve()
else:
    raise FileNotFoundError(f"Cannot find database/data/ from {_cwd}")

# (depth, nc_time_suffix, DT_H_hours, time_label)
# DT_H: 00-07d = 7*24=168h, 07-14d = 7*24=168h, 07-28d = 21*24=504h
FILES = [
    ("05m", "00-07days", 168,  "00d-07d"),
    ("05m", "07-14days", 168,  "07d-14d"),
    ("05m", "07-28days", 504,  "07d-28d"),
    ("10m", "00-07days", 168,  "00d-07d"),
    ("10m", "07-14days", 168,  "07d-14d"),
    ("10m", "07-28days", 504,  "07d-28d"),
    ("15m", "00-07days", 168,  "00d-07d"),
    ("15m", "07-14days", 168,  "07d-14d"),
    ("15m", "07-28days", 504,  "07d-28d"),
]

_TIME_TO_DAYS = {"00-07days": "07", "07-14days": "14", "07-28days": "28"}

ESCAPE_HEX_STR = "(0, 0, 0)"

MIN_CLUSTER_SIZE = 4   # BFS: remove components with fewer hexes
WEIGHT_SIG_FIGS  = 3   # significant figures for weight rounding

HEX_VARS = [
    "water_fraction",
    "depth_mean", "depth_median", "depth_std",
    "aqc_count", "rst_count", "pop_count",
    "dss_count", "hly_count", "his_count",
    "lon", "lat",
]

def local_path(depth, time_suffix):
    days = _TIME_TO_DAYS[time_suffix]
    return LOCAL_INPUT_DIR / f"{depth}_ds_conn_{days}.nc"

def _to_str(v):
    return v.decode() if isinstance(v, bytes) else str(v)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"LOCAL_INPUT_DIR: {LOCAL_INPUT_DIR}")
print(f"OUT_DIR:         {OUT_DIR}")
print(f"Files exist: {[local_path(d,t).exists() for d,t,_,_ in FILES]}")

LOCAL_INPUT_DIR: /Users/wrath/src/github.com/geomar-od-lagrange/ostrea/preproc/input_comp
OUT_DIR:         /Users/wrath/src/github.com/geomar-od-lagrange/ostrea/database/data
Files exist: [True, True, True, True, True, True, True, True, True]


In [2]:
import gc
import json
import math
import time as time_mod
from collections import Counter

import numpy as np
import pandas as pd
import xarray as xr

## Phase 1: Build canonical hex metadata

Open each NC file, read only the small per-hex arrays (NOT `obs`).
Collect union of hex0 + hex1 labels across all 9 files.
First-seen-wins for metadata (identical across files for the same hex).

In [3]:
hex_data = {}  # label_str -> {var: value, 'lon_corners': [...], 'lat_corners': [...]}

for depth, time_suffix, _, _ in FILES:
    p = local_path(depth, time_suffix)
    t0 = time_mod.time()
    ds = xr.open_dataset(p, engine="netcdf4")

    for dim in ["hex0", "hex1"]:
        labels = ds[dim].values
        for i, raw in enumerate(labels):
            lbl = _to_str(raw)
            if lbl == ESCAPE_HEX_STR:
                continue
            if lbl in hex_data:
                continue  # first-seen-wins
            rec = {}
            for v in HEX_VARS:
                key = f"{v}_{dim}"
                if key in ds:
                    rec[v] = float(ds[key].values[i])
            for coord in ["lon", "lat"]:
                key = f"{coord}_{dim}_corners"
                if key in ds:
                    rec[f"{coord}_corners"] = ds[key].values[:, i].tolist()
            hex_data[lbl] = rec

    ds.close()
    print(f"{depth} {time_suffix}: {len(hex_data)} unique hexes  ({time_mod.time()-t0:.1f}s)")

print(f"\nTotal unique hexes (excl. escape): {len(hex_data)}")

05m 00-07days: 8397 unique hexes  (1.8s)
05m 07-14days: 8397 unique hexes  (0.0s)
05m 07-28days: 8397 unique hexes  (0.0s)
10m 00-07days: 8413 unique hexes  (0.0s)
10m 07-14days: 8413 unique hexes  (0.0s)
10m 07-28days: 8413 unique hexes  (0.0s)
15m 00-07days: 8425 unique hexes  (0.0s)
15m 07-14days: 8425 unique hexes  (0.0s)


15m 07-28days: 8425 unique hexes  (0.0s)

Total unique hexes (excl. escape): 8425


In [4]:
# Deterministic: sort label strings lexicographically → assign IDs 0..N-1
sorted_labels = sorted(hex_data.keys())
label_to_id = {lbl: i for i, lbl in enumerate(sorted_labels)}

print(f"ID range: 0 – {len(label_to_id) - 1}")
print(f"First 3: {sorted_labels[:3]}")
print(f"Last  3: {sorted_labels[-3:]}")

ID range: 0 – 8424
First 3: ['(-1, -19, 20)', '(-1, -2, 3)', '(-1, -20, 21)']
Last  3: ['(9, 7, -16)', '(9, 8, -17)', '(9, 9, -18)']


In [5]:
mapping_path = OUT_DIR / "hex_label_to_id.json"
with open(mapping_path, "w") as f:
    json.dump(label_to_id, f)
print(f"Written: {mapping_path} ({len(label_to_id)} entries)")

Written: /Users/wrath/src/github.com/geomar-od-lagrange/ostrea/database/data/hex_label_to_id.json (8425 entries)


In [6]:
features = []
for lbl in sorted_labels:
    hex_id = label_to_id[lbl]
    rec = hex_data[lbl]
    lons = rec.get("lon_corners", [])
    lats = rec.get("lat_corners", [])
    coords = [[lo, la] for lo, la in zip(lons, lats)]
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    features.append({
        "type": "Feature",
        "properties": {"id": hex_id},
        "geometry": {"type": "Polygon", "coordinates": [coords]},
    })

geojson = {"type": "FeatureCollection", "features": features}
out_path = OUT_DIR / "hexes.geojson"
with open(out_path, "w") as f:
    json.dump(geojson, f)
print(f"Written: {out_path} ({len(features)} features)")

Written: /Users/wrath/src/github.com/geomar-od-lagrange/ostrea/database/data/hexes.geojson (8425 features)


In [7]:
col_map = {
    "id":             lambda lbl, rec: label_to_id[lbl],
    "lon":            lambda lbl, rec: rec.get("lon"),
    "lat":            lambda lbl, rec: rec.get("lat"),
    "depth":          lambda lbl, rec: rec.get("depth_median"),
    "water_fraction": lambda lbl, rec: rec.get("water_fraction"),
    "disease":        lambda lbl, rec: rec.get("dss_count", 0.0),
    "rest":           lambda lbl, rec: rec.get("rst_count",  0.0),
    "aqc":            lambda lbl, rec: rec.get("aqc_count",  0.0),
    "pop":            lambda lbl, rec: rec.get("pop_count",  0.0),
    "his":            lambda lbl, rec: rec.get("his_count",  0.0),
    "hly":            lambda lbl, rec: rec.get("hly_count",  0.0),
}

rows = {col: {} for col in col_map}
for lbl in sorted_labels:
    rec = hex_data[lbl]
    hex_id = label_to_id[lbl]
    for col, fn in col_map.items():
        rows[col][str(hex_id)] = fn(lbl, rec)

out_path = OUT_DIR / "meta.json"
with open(out_path, "w") as f:
    json.dump(rows, f)
print(f"Written: {out_path} ({len(rows['id'])} entries, columns: {list(rows.keys())})")

Written: /Users/wrath/src/github.com/geomar-od-lagrange/ostrea/database/data/meta.json (8425 entries, columns: ['id', 'lon', 'lat', 'depth', 'water_fraction', 'disease', 'rest', 'aqc', 'pop', 'his', 'hly'])


## Phase 2: Compute connectivity

Process each NC file sequentially. For each file:
1. Load full `obs` array → `(month, year, hex0, hex1)`
2. Sum over month+year → `obs_sum`
3. Delete `obs` immediately to free memory
4. Compute normalized weight F per source→target pair
5. Accumulate records, tagged with depth + time_label

**F formula** (normalized relative dilution factor):
```
F(i,j) = [obs_sum(i,j) / (N_hex0_sum(i) * DT_H * n_months_years)] * [wf_hex0(i) / wf_hex1(j)]
```
Rounded to `WEIGHT_SIG_FIGS` significant figures; zero-after-rounding rows dropped.

In [8]:
all_records = []

for depth, time_suffix, DT_H, time_label in FILES:
    p = local_path(depth, time_suffix)
    t0 = time_mod.time()
    print(f"\n--- {depth} {time_suffix} (DT_H={DT_H}h) ---")

    ds = xr.open_dataset(p, engine="netcdf4")

    # Small metadata arrays
    hex0_labels = ds["hex0"].values
    hex1_labels = ds["hex1"].values
    wf_hex0 = ds["water_fraction_hex0"].values
    wf_hex1 = ds["water_fraction_hex1"].values
    n_hex0 = ds.sizes["hex0"]
    n_months_years = ds.sizes["month"] * ds.sizes["year"]

    hex0_str = np.array([_to_str(b) for b in hex0_labels])
    hex1_str = np.array([_to_str(b) for b in hex1_labels])
    escape_mask = np.array([s == ESCAPE_HEX_STR for s in hex1_str])
    valid_hex1  = ~escape_mask
    hex1_ids    = np.array([label_to_id.get(s, -1) for s in hex1_str], dtype=np.int64)

    # Load full obs — shape (month, year, hex0, hex1)
    print(f"  Loading obs ...", end=" ", flush=True)
    obs_all  = ds["obs"].values
    ds.close()
    print(f"done  shape={obs_all.shape}")

    obs_sum     = np.nansum(obs_all, axis=(0, 1))   # (hex0, hex1)
    N_hex0_sum  = obs_sum.sum(axis=1)               # (hex0,)
    del obs_all
    gc.collect()

    file_records = []
    for i in range(n_hex0):
        if i % 2000 == 0:
            print(f"  {i}/{n_hex0} ...", flush=True)
        if N_hex0_sum[i] == 0:
            continue
        src_id = label_to_id.get(hex0_str[i], -1)
        if src_id == -1:
            continue
        wf0 = wf_hex0[i]
        if wf0 == 0 or not math.isfinite(wf0):
            continue

        row = obs_sum[i]
        # valid targets: not escape, obs > 0, known ID, wf > 0
        tmask = valid_hex1 & (row > 0) & (hex1_ids >= 0) & (wf_hex1 > 0)
        if not tmask.any():
            continue

        tgt_ids  = hex1_ids[tmask]
        tgt_obs  = row[tmask]
        tgt_wf1  = wf_hex1[tmask]

        F_raw = (tgt_obs / (N_hex0_sum[i] * DT_H * n_months_years)) * (wf0 / tgt_wf1)
        exp   = np.floor(np.log10(F_raw))
        scale = 10.0 ** (WEIGHT_SIG_FIGS - 1 - exp)
        F     = np.round(F_raw * scale) / scale

        keep = F > 0
        if not keep.any():
            continue

        file_records.append(pd.DataFrame({
            "start_id": np.int64(src_id),
            "end_id":   tgt_ids[keep],
            "weight":   F[keep],
        }))

    del obs_sum, N_hex0_sum
    gc.collect()

    if file_records:
        df_file = pd.concat(file_records, ignore_index=True)
        df_file["depth"] = depth
        df_file["time"]  = time_label
        all_records.append(df_file)
        print(f"  -> {len(df_file):,} rows in {time_mod.time()-t0:.0f}s")
    else:
        print(f"  -> 0 rows (!)")

print(f"\nPhase 2 done.")


--- 05m 00-07days (DT_H=168h) ---
  Loading obs ... 

done  shape=(5, 4, 8364, 8397)


  0/8364 ...


  2000/8364 ...


  4000/8364 ...


...

  2000/8223 ...


  4000/8223 ...


  6000/8223 ...


  8000/8223 ...


  -> 3,215,520 rows in 17s

Phase 2 done.


In [9]:
conn = pd.concat(all_records, ignore_index=True)
conn = conn[["start_id", "end_id", "time", "depth", "weight"]]

print(f"Total rows: {len(conn):,}")
print(f"Dtypes:\n{conn.dtypes}")
print(f"Weight range: {conn.weight.min():.3e} – {conn.weight.max():.3e}")
print()
for (d, t), g in conn.groupby(["depth", "time"]):
    print(f"  {d} {t}: {len(g):>10,} rows")

Total rows: 18,530,749
Dtypes:
start_id      int64
end_id        int64
time         object
depth        object
weight      float64
dtype: object
Weight range: 3.310e-11 – 5.480e-04



  05m 00d-07d:  1,028,496 rows
  05m 07d-14d:  1,892,431 rows
  05m 07d-28d:  3,407,859 rows
  10m 00d-07d:  1,015,297 rows
  10m 07d-14d:  1,861,171 rows
  10m 07d-28d:  3,320,873 rows
  15m 00d-07d:    985,539 rows
  15m 07d-14d:  1,803,563 rows
  15m 07d-28d:  3,215,520 rows


In [10]:
# Write 9 parquets (pre-pruning; will be overwritten in Phase 3)
for (depth, time_label), group in conn.groupby(["depth", "time"]):
    out_path = OUT_DIR / f"connectivity_{depth}_{time_label}.pq"
    group[["start_id", "end_id", "time", "depth", "weight"]].to_parquet(out_path, index=False)
    print(f"Written (pre-prune): {out_path.name}  ({len(group):,} rows)")

Written (pre-prune): connectivity_05m_00d-07d.pq  (1,028,496 rows)


Written (pre-prune): connectivity_05m_07d-14d.pq  (1,892,431 rows)


Written (pre-prune): connectivity_05m_07d-28d.pq  (3,407,859 rows)
Written (pre-prune): connectivity_10m_00d-07d.pq  (1,015,297 rows)


Written (pre-prune): connectivity_10m_07d-14d.pq  (1,861,171 rows)


Written (pre-prune): connectivity_10m_07d-28d.pq  (3,320,873 rows)
Written (pre-prune): connectivity_15m_00d-07d.pq  (985,539 rows)


Written (pre-prune): connectivity_15m_07d-14d.pq  (1,803,563 rows)


Written (pre-prune): connectivity_15m_07d-28d.pq  (3,215,520 rows)


## Phase 3: Prune disconnected hex clusters

**Stage A**: Remove hexes whose corner coordinates contain NaN or Inf.

**Stage B**: BFS on cube-coordinate adjacency (not the transport graph).
Remove connected components with fewer than `MIN_CLUSTER_SIZE` hexes.

IDs are **not** renumbered — sparse IDs are preserved throughout.

In [11]:
# Stage A: NaN/Inf corners
with open(OUT_DIR / "hexes.geojson") as f:
    gj = json.load(f)

bad_ids = set()
for feat in gj["features"]:
    coords = feat["geometry"]["coordinates"][0]
    if any(not math.isfinite(v) for pt in coords for v in pt):
        bad_ids.add(feat["properties"]["id"])

print(f"Hexes with NaN/Inf corners: {len(bad_ids)}  ids={sorted(bad_ids)}")

Hexes with NaN/Inf corners: 11  ids=[33, 1974, 1985, 2191, 5519, 5605, 5668, 7356, 7563, 8048, 8135]


In [12]:
# Stage B: BFS spatial connected-component pruning
def parse_cube(label: str):
    return tuple(int(x) for x in label.strip("()").split(","))

DIRECTIONS = [(1,-1,0),(-1,1,0),(1,0,-1),(-1,0,1),(0,1,-1),(0,-1,1)]

# Build cube→id map, excluding bad_ids already
cubes = {
    parse_cube(lbl): int(hex_id)
    for lbl, hex_id in label_to_id.items()
    if int(hex_id) not in bad_ids
}
cube_set = set(cubes)

visited    = set()
components = []

for cube in cube_set:
    if cube in visited:
        continue
    component = set()
    queue = [cube]
    while queue:
        c = queue.pop()
        if c in visited:
            continue
        visited.add(c)
        component.add(cubes[c])
        for d in DIRECTIONS:
            nb = (c[0]+d[0], c[1]+d[1], c[2]+d[2])
            if nb in cube_set and nb not in visited:
                queue.append(nb)
    components.append(component)

components.sort(key=len)
size_counts = Counter(len(c) for c in components)
print(f"Total components: {len(components)}")
print(f"Size distribution:")
for size in sorted(size_counts):
    print(f"  {size:4d} hex: {size_counts[size]:3d} component(s)")

small_ids = set().union(*(c for c in components if len(c) < MIN_CLUSTER_SIZE))
print(f"\nHex IDs to prune (size < {MIN_CLUSTER_SIZE}): {len(small_ids)}")

Total components: 3
Size distribution:
     1 hex:   1 component(s)
     3 hex:   1 component(s)
  8410 hex:   1 component(s)

Hex IDs to prune (size < 4): 4


In [13]:
prune_ids = bad_ids | small_ids
print(f"Total prune_ids: {len(prune_ids)} (bad_corners={len(bad_ids)}, small_cluster={len(small_ids)})")

# hexes.geojson
before = len(gj["features"])
gj["features"] = [f for f in gj["features"] if f["properties"]["id"] not in prune_ids]
with open(OUT_DIR / "hexes.geojson", "w") as f:
    json.dump(gj, f)
print(f"hexes.geojson:         {before} -> {len(gj['features'])} features")

# meta.json
with open(OUT_DIR / "meta.json") as f:
    meta = json.load(f)
bad_row_keys = {k for k, v in meta["id"].items() if v in prune_ids}
before_meta = len(meta["id"])
meta = {col: {k: v for k, v in col_data.items() if k not in bad_row_keys}
        for col, col_data in meta.items()}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f)
print(f"meta.json:             {before_meta} -> {len(meta['id'])} entries")

# hex_label_to_id.json
before_l2id = len(label_to_id)
label_to_id_pruned = {lbl: i for lbl, i in label_to_id.items() if i not in prune_ids}
with open(OUT_DIR / "hex_label_to_id.json", "w") as f:
    json.dump(label_to_id_pruned, f)
print(f"hex_label_to_id.json:  {before_l2id} -> {len(label_to_id_pruned)} entries")

# connectivity parquets
for path in sorted(OUT_DIR.glob("connectivity_*.pq")):
    df = pd.read_parquet(path)
    before_pq = len(df)
    mask = df["start_id"].isin(prune_ids) | df["end_id"].isin(prune_ids)
    df = df[~mask].copy()
    df.to_parquet(path, index=False)
    print(f"{path.name}: {before_pq:>10,} -> {len(df):>10,} rows ({mask.sum()} dropped)")

Total prune_ids: 15 (bad_corners=11, small_cluster=4)


hexes.geojson:         8425 -> 8410 features


meta.json:             8425 -> 8410 entries
hex_label_to_id.json:  8425 -> 8410 entries


connectivity_05m_00d-07d.pq:  1,028,496 ->  1,028,478 rows (18 dropped)


connectivity_05m_07d-14d.pq:  1,892,431 ->  1,892,423 rows (8 dropped)


connectivity_05m_07d-28d.pq:  3,407,859 ->  3,407,838 rows (21 dropped)
connectivity_10m_00d-07d.pq:  1,015,297 ->  1,015,297 rows (0 dropped)


connectivity_10m_07d-14d.pq:  1,861,171 ->  1,861,171 rows (0 dropped)


connectivity_10m_07d-28d.pq:  3,320,873 ->  3,320,873 rows (0 dropped)
connectivity_15m_00d-07d.pq:    985,539 ->    985,535 rows (4 dropped)


connectivity_15m_07d-14d.pq:  1,803,563 ->  1,803,558 rows (5 dropped)


connectivity_15m_07d-28d.pq:  3,215,520 ->  3,215,515 rows (5 dropped)


## Verification

In [14]:
import sys

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
failures = []

# 1. File existence
for fname in ["hex_label_to_id.json", "hexes.geojson", "meta.json"]:
    ok = (OUT_DIR / fname).exists()
    print(f"  {'OK' if ok else 'MISSING'}: {fname}")
    if not ok: failures.append(f"Missing {fname}")

for depth, _, _, time_label in FILES:
    pq = OUT_DIR / f"connectivity_{depth}_{time_label}.pq"
    ok = pq.exists()
    print(f"  {'OK' if ok else 'MISSING'}: {pq.name}")
    if not ok: failures.append(f"Missing {pq.name}")

# 2. Load final metadata
with open(OUT_DIR / "hex_label_to_id.json") as f:
    final_l2id = json.load(f)
with open(OUT_DIR / "meta.json") as f:
    final_meta = json.load(f)
with open(OUT_DIR / "hexes.geojson") as f:
    final_gj = json.load(f)

# 3. ID set consistency
id_set_l2id = set(final_l2id.values())
id_set_meta = set(final_meta["id"].values())
id_set_gj   = {feat["properties"]["id"] for feat in final_gj["features"]}

if id_set_l2id == id_set_meta == id_set_gj:
    print(f"\n  OK: ID sets consistent across all 3 metadata files ({len(id_set_l2id)} hexes)")
else:
    msg = f"ID set mismatch: l2id={len(id_set_l2id)}, meta={len(id_set_meta)}, geojson={len(id_set_gj)}"
    print(f"  FAIL: {msg}")
    failures.append(msg)

# 4. No escape hex
if ESCAPE_HEX_STR not in final_l2id:
    print(f"  OK: escape hex absent from label_to_id")
else:
    failures.append("escape hex still in label_to_id")

# 5. No NaN/Inf corners
bad_corner_count = 0
for feat in final_gj["features"]:
    coords = feat["geometry"]["coordinates"][0]
    if any(not math.isfinite(v) for pt in coords for v in pt):
        bad_corner_count += 1
if bad_corner_count == 0:
    print(f"  OK: no NaN/Inf corners in hexes.geojson")
else:
    failures.append(f"{bad_corner_count} features with NaN/Inf corners remain")

# 6. Connectivity checks
print()
print(f"  {'File':<35} {'Rows':>10}  {'w_min':>10}  {'w_max':>10}  Status")
print(f"  {'-'*35} {'-'*10}  {'-'*10}  {'-'*10}  ------")
total_rows = 0
for depth, _, _, time_label in FILES:
    df  = pd.read_parquet(OUT_DIR / f"connectivity_{depth}_{time_label}.pq")
    ok  = True
    end_ids_int = df["end_id"].astype(np.int64)
    if not (df["weight"] > 0).all():
        failures.append(f"zero/neg weight in {depth}_{time_label}"); ok = False
    if not (df["weight"] < 1).all():
        failures.append(f"weight >= 1 in {depth}_{time_label}"); ok = False
    unknown_start = set(df["start_id"].unique()) - id_set_l2id
    unknown_end   = set(end_ids_int.unique())    - id_set_l2id
    if unknown_start:
        failures.append(f"unknown start_ids in {depth}_{time_label}: {unknown_start}"); ok = False
    if unknown_end:
        failures.append(f"unknown end_ids in {depth}_{time_label}: {unknown_end}"); ok = False
    total_rows += len(df)
    status = "OK" if ok else "FAIL"
    print(f"  connectivity_{depth}_{time_label}.pq  {len(df):>10,}  {df.weight.min():>10.2e}  {df.weight.max():>10.2e}  {status}")

print(f"\n  Total connectivity rows: {total_rows:,}")
if total_rows < 15_000_000:
    failures.append(f"total rows {total_rows:,} < 15M")

print()
if failures:
    print(f"FAILURES ({len(failures)}):")
    for msg in failures:
        print(f"  - {msg}")
    sys.exit(1)
else:
    print("All checks PASSED.")

VERIFICATION
  OK: hex_label_to_id.json
  OK: hexes.geojson
  OK: meta.json
  OK: connectivity_05m_00d-07d.pq
  OK: connectivity_05m_07d-14d.pq
  OK: connectivity_05m_07d-28d.pq
  OK: connectivity_10m_00d-07d.pq
  OK: connectivity_10m_07d-14d.pq
  OK: connectivity_10m_07d-28d.pq
  OK: connectivity_15m_00d-07d.pq
  OK: connectivity_15m_07d-14d.pq
  OK: connectivity_15m_07d-28d.pq

  OK: ID sets consistent across all 3 metadata files (8410 hexes)
  OK: escape hex absent from label_to_id
  OK: no NaN/Inf corners in hexes.geojson

  File                                      Rows       w_min       w_max  Status
  ----------------------------------- ----------  ----------  ----------  ------
  connectivity_05m_00d-07d.pq   1,028,478    2.65e-10    3.37e-04  OK
  connectivity_05m_07d-14d.pq   1,892,423    2.65e-10    4.32e-04  OK


  connectivity_05m_07d-28d.pq   3,407,838    3.38e-11    9.92e-05  OK
  connectivity_10m_00d-07d.pq   1,015,297    2.63e-10    3.42e-04  OK
  connectivity_10m_07d-14d.pq   1,861,171    2.63e-10    5.30e-04  OK
  connectivity_10m_07d-28d.pq   3,320,873    3.34e-11    1.63e-04  OK


  connectivity_15m_00d-07d.pq     985,535    2.61e-10    3.95e-04  OK
  connectivity_15m_07d-14d.pq   1,803,558    2.61e-10    5.48e-04  OK
  connectivity_15m_07d-28d.pq   3,215,515    3.31e-11    1.91e-04  OK

  Total connectivity rows: 18,530,688

All checks PASSED.
